In [ ]:
# MUlti agenteams ---- Researcher and Writer

"""
Today I stop thinking of my graph as a single agent doing everything and start thinking 
in roles. Im going to build two specialist agents- A Researcher whose only job is to retrieve
and structure evidence from the Apple 10-K, and a writer whose only job is to turn that evidence
into a polished, well-structured report. They communicate by passing structured data through
shared state, and the Writer can send a request back to the Researcher if it needs more information
making this a genuine back-and-forth multi-agent collaboration, not just two sequential nodes.

Deep coding: Researcher + Writer, feedback loop
The Researcher Agent: A node that receives a topic, runs multiple targeted searches against my Apple 10-K
vector store, and returns a structured ResearchPacket--- A dictionary containing the topic, a list
of retrieved evidence chunks with their sources and confidence scores, and a coverage
assessment("did i find enough to write about this?") 
The Writer Agent: A node that receives the ResearchPacket and produces a polished report section-
not just an answer but a properly structured piece of writing with a heading, body paragraphs
, and a data table where relevant. Crucially, the Writer also outputs a needs_more_research
flag and a specific follow_up_query if the evidence feels thin.
The FeedbackLoop: A conditional edge after the writer node - if needs_more_research is True and
attempts remain, route back to the Researcher with the follow_up_query. If satisfied or
attempts exhausted, route to a compile_node that assembles all report sections into a final document.
The Compile Node: Receives all accumulated report sections from multiple Researcher/Writer cycles and assembles them
into one coherent document with an executive summary prepended.

"""

'\nToday i stop thinking of my graph as a single agent doing everything and start thinking \nin roles. Im going to build two specialist agents- A Researcher whose only job is to retrieve\nand structure evidence from the Apple 10-K, and a writer whose only job is to turn that evidence\ninto a polished, well-structured report. They communicate by passing structured data through\nshared state, and the Writer can send a request back to the Researcher if it needs more information\nmaking this a genuine back-and-forth multi-agent collaboration, not just two sequential nodes.\n\nDeep coding: Researcher + Writer, feedback loop\nThe Researcher Agent: A node that receives a topic, runs multiple targeted searches against my Apple 10-K\nvector store, and returns a structured ResearchPacket--- A dictionary containing the topic, a list\nof retrieved evidence chunks with their sources and confidence scores, and a coverage\nassessment("did i find enough to write about this?")\nThe Writer Agent: A node

In [2]:

# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"
    

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

   # response = client.chat.completions.create(
      #  model = "openai/gpt-oss-120b",
        messages = [
            #{"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ]#,

       # temperature= 0,
   # )

    return tracked_llm_call(messages= messages, system= GENERATOR_SYSTEM_PROMPT)


def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    
    messages = [
        #{"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
        {"role": "user", "content":(
            f"Question: {question}\n\n"
            f"Source Context:\n{context}\n\n"
            f"Generated Answer:\n{answer}"

        )}
    ]

    raw = tracked_llm_call(messages= messages, system= REVIEWER_SYSTEM_PROMPT)
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason

# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }


# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result



c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_18436\2373246763.py:30: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


🤖🛩️ Vector Store connection established ⚡


In [3]:
# importing necessary libraries for today

import uuid
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# improving reused RAG state from day 16

# THE STATE SCHEMA

class RAGState(TypedDict):
    # Input
    question: str

    # Router output
    classification: Optional[str]
    classification_reason: Optional[str]

    # Retrieval output
    context: Optional[str]
    retrieval_source: Optional[str]
    retrieval_score: Optional[float]

    # Generation output
    answer: Optional[str]

    # Reviewer output
    reviewer_verdict: Optional[str]
    reviewer_reason: Optional[str]

    # Rewrite tracking
    rewrite_count: int
    max_rewrites: int

    # Final flags
    warning: Optional[str]
    finished_at: Optional[str]

    # Sufficiency and reformulation tracking
    sufficiency_verdict: Optional[str]
    sufficiency_reason: Optional[str]
    retrieval_attempts: int
    max_retrieval_attempts: int
    reformulated_query: Optional[str]

    # NEW today - human approval tracking
    human_approved: Optional[bool]
    approval_reason: Optional[str]
    action_taken: Optional[str]






# THE FIVE NODES

# -----Node 1: Classify -------
def classify_node(state: RAGState) -> dict:
    print(f"\n[NODE: classify] Question: {state['question'][:60]}...")
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Question: {state['question']}"}],
        system= ROUTER_SYSTEM_PROMPT
    )

    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not parse."

    print(f" -> Classification: {classification} - {reason}")
    return {"classification": classification, "classification_reason": reason}

# ------ Node 2: Retrieve ------ # LEAB=VING THIS ONE OUT AS WE ARE GOING TO WRITE A MODIFIED VERSION OF IT IN THE NEXT CELL


# ---Node 3: Generate ----
def generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: generate]")

    if state["classification"] == "UNKNOWN":
        answer = (
            "This question cannot be answered using the Apple FY2024 10-K document"
            "or available tools."
        )
        print(f" -> UNKNOWN path - returning abstention")
        return {"answer": answer}
    
    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    answer = tracked_llm_call(
        messages = [{
            "role":"user",
            "content": (
                f"Context:\n{state['context']}\n\n"
                f"Question: {state['question']}"
                f"{critique_block}"
            )

        }],
        system= GENERATOR_SYSTEM_PROMPT
    )
    print(f" -> Answer generated ({len(answer)} chars)")
    return {"answer": answer}


# -----Node 4: Review ----
def review_node(state: RAGState) -> dict:
    print(f"\n[NODE: review]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN path - skipping review")
        return {"reviewer_verdict": "PASS", "reviewer_reason": "Abstention accepted."}
    
    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Source Context:\n{state['context']}\n\n"
                f"Generated Answer:\n{state['answer']}"
            )
        }],
        system = REVIEWER_SYSTEM_PROMPT
    )

    verdict_match =re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Reviewer: {verdict} - {reason}")
    return {"reviewer_verdict" : verdict, "reviewer_reason": reason}


# ---- Node 5: Rewrite ----
def rewrite_node(state: RAGState) -> dict:
    new_count = state.get("rewrite_count", 0) + 1
    print(f"\n[NODE: rewrite] Attempt {new_count}")
    return {
        "rewrite_count" : new_count,
        "answer": None # cleared so generate_node produces a fresh answer
    }


# CONDITIONAL EDGE LOGIC

def route_after_review(state: RAGState) -> Literal["rewrite_node", "__end__"]:
    """
    Called after review_node .
    Decides whether to loop back for a rewrite or proceed to END.

    """

    verdict = state.get("reviewer_verdict", "FAIL")
    rewrite_count = state.get("rewrite_count", 0)
    max_rewrites = state.get("max_rewrites", 2)

    if verdict == "PASS":
        print(" -> Edge: PASS -> END")
        return "__end__"
    
    if rewrite_count >= max_rewrites:
        print(f" -> Edge: max rewrites ({max_rewrites}) reached -> END with warning")
        return "__end__"
    
    print(f" -> Edge: FAIL -> rewrite_node (attempt {rewrite_count + 1})")
    return "rewrite_node"


    

# MODIFIED RETRIEVE NODE FROM DAY 14
# the only difference here is it checks for a reformulated query before falling back to the original question

def retrieve_node(state: RAGState) -> dict:
    print(f"\n[NODE: retrieve]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN query - skipping retrieval")
        return{
            "context": "No context retrieved - query classified as UNKNOWN.",
            "retrieval_source": "none",
            "retrieval_score": 0.0

        }
    # Use reformulated query if one exists, otherwise the original question
    search_query = state.get("reformulated_query") or state["question"]
    print(f" -> Searching with: {search_query[: 80]}")
    
    docs, score = retrieve_with_confidence(search_query)

    if score < RELEVANCE_THRESHOLD or not docs:
        print(" -> CRAG: Low confidence - falling back to web search")
        web_results = webSearch.invoke(search_query)
        context = "\n\n".join([r["content"] for r in web_results])
        return {"context": context, "retrieval_source": "web", "retrieval_score": score}
    
    context = format_chunks(docs= docs)
    print(f" -> Local retrieval accepted (score: {score:.3f})")
    return {"context": context, "retrieval_source": "local", "retrieval_score": score}

# THE SUFFICIENCY CHECKER NODE

SUFFICIENCY_SYSTEM_PROMPT = """You are a retrieval sufficiency checker for a financial RAG system.

You will receive a question and a block of retrieved context.
Your job is to determine ONLY whether the context contains enough information
to answer the question - do not answer the question itself.

Respond in exactly this format:
Sufficiency: <SUFFICIENT or INSUFFICIENT>
Reason: <one sentence explaining why>

SUFFICIENT means the context directly contains the facts needed to answer.
INSUFFICIENT means the context is missing key facts, is off-topic, or only partially covers the question.
"""

def check_sufficiency_node(state: RAGState) -> dict:
    print(f"\n[NODE: check_sufficiency]")

    if state["classification"] == "UNKNOWN" or state.get("retrieval_source") == "web":
        # web fallback already hapened - let CRAG's existing logic handle it downstream
        print(" -> Skipping sufficiency check (UNKNOWN or already on web fallback)")
        return {"sufficiency_verdict": "SUFFICIENT", "sufficiency_reason": "Bypassed."}
    
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Retrieved Context:\n{state['context']}"
            )
        }],
        system= SUFFICIENCY_SYSTEM_PROMPT


    )

    verdict_match = re.search(r"Sufficiency:\s*(SUFFICIENT|INSUFFICIENT)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "INSUFFICIENT"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Sufficiency: {verdict} - {reason}")
    return {"sufficiency_verdict": verdict, "sufficiency_reason": reason}

# QUERY REFORMULATION NODE
REFORMULATE_SYSTEM_PROMPT = """You are a search query optimiser for a financial RAG system.

The previous search query did not retrieve sufficient context to answer the question.
Rewrite the search query to improve retrieval - use broader terms, different financial
terminology, or break the question into a more specific sub-question.

Respond with ONLY the new search query, nothing else. No explanation, no preamble.

"""

def reformulate_node(state: RAGState) -> dict:
    new_attempts = state.get("retrieval_attempts", 0) + 1
    print(f"\n[NODE: reformulate] Generating new query (will become attempt {new_attempts + 1})")

    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Original question: {state['question']}\n\n"
                f"Previous search query that failed: "
                f"{state.get('reformulated_query') or state['question']}\n\n"
                f"Why it failed: {state.get('sufficiency_reason', 'Context was insufficient.')}"

            )
        }],
        system= REFORMULATE_SYSTEM_PROMPT
    )

    new_query = raw.strip()
    print(f" -> New query: {new_query}")

    return {
        "reformulated_query": new_query,
        "retrieval_attempts": new_attempts
    }

# CONDITIONAL EDGE AFTER SUFFICIENCY CHECK

def route_after_sufficiency(state: RAGState) -> Literal["generate_node", "reformulate_node"]:
    """
    Decides whether to proceed to generation or loop back for a better search.

    """
    verdict = state.get("sufficiency_verdict", "SUFFICIENT")
    attempts = state.get("retrieval_attempts", 0)
    max_attempts = state.get("max_retrieval_attempts", 2)

    if verdict == "SUFFICIENT":
        print(" -> Edge: SUFFCIENT -> generate_node")
        return "generate_node"
    
    if attempts >= max_attempts:
        print(f" -> Edge: max retrieval attempts ({max_attempts}) reached -> generate_node anyway")
        return "generate_node"
    
    print(f" -> Edge: INSUFFICIENT -> reformulate_node (attempt {attempts + 1})")
    return "reformulate_node"

DB_PATH = "checkpoints/rag_checkpoints.db"


In [4]:
# Extended state schema

class MultiAgentState(TypedDict):
    # Input
    topic: str
    report_sections_requested: list[str]

    # Researcher outputs
    current_research_query: Optional[str]
    research_packets: list[dict]       # one per Researcher call
    researcher_attempts: int
    max_researcher_attempts: int

    # Writer outputs
    report_sections: list[str]         # accumulates across cycles
    needs_more_research: bool
    follow_up_query: Optional[str]
    writer_attempts: int

    # Compile output
    final_report: Optional[str]
    finished_at: Optional[str]

In [36]:
# the researcher agent node

RESEARCHER_SYSTEM_PROMPT = """You are a specialist financial researcher working on
an Apple Inc. FY2024 10-K analysis.

You will receive a research query. Your job is to:
1. Identify the 3 most important facts, figures, or statements that answer this query
2. For each piece of evidence, note exactly what it says and how confident you are

Respond in EXACTLY this format:

EVIDENCE 1:
Fact: <the specific fact or figure>
Confidence: <HIGH, MEDIUM, or LOW>

EVIDENCE 2:
Fact: <the specific fact or figure>
Confidence: <HIGH, MEDIUM, or LOW>

EVIDENCE 3:
Fact: <the specific fact or figure>
Confidence: <HIGH, MEDIUM, or LOW>

COVERAGE: <COMPLETE, PARTIAL, or INSUFFICIENT>
COVERAGE_REASON: <one sentence explaining why>
"""

def researcher_node(state: MultiAgentState) -> dict:
    query = state.get("follow_up_query") or state.get("current_research_query") or state["topic"]
    attempts = state.get("researcher_attempts", 0) + 1

    print(f"\n[NODE: researcher] Attempt {attempts}")
    print(f"  → Query: {query[:80]}")

    # Retrieve from Apple 10-K vector store
    retriever = index.as_retriever(similarity_top_k = 4)
    docs = retriever.retrieve(query)

    # Calculate the score manually from the first document (if it exists)
    score = docs[0].score if docs and docs[0].score is not None else 0.0




    if not docs:
        context = "No relevant content found in the Apple FY2024 10-K for this query."
    else:
        context = "\n\n---\n\n".join([doc.node.get_content() for doc in docs])

    print(f"  → Retrieved {len(docs)} chunks (top score: {score:.3f})")

    # Researcher LLM call — structures the evidence
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Research Query: {query}\n\n"
                f"Retrieved Context from Apple FY2024 10-K:\n{context}"
            )
        }],
        system=RESEARCHER_SYSTEM_PROMPT
    )

    # Parse the structured evidence
    evidence_blocks = re.findall(
        r"EVIDENCE \d+:\nFact: (.+?)\nConfidence: (HIGH|MEDIUM|LOW)",
        raw,
        re.DOTALL
    )
    coverage_match = re.search(r"COVERAGE: (COMPLETE|PARTIAL|INSUFFICIENT)", raw)
    coverage_reason_match = re.search(r"COVERAGE_REASON: (.+)", raw)

    evidence_list = [
        {"fact": fact.strip(), "confidence": conf}
        for fact, conf in evidence_blocks
    ]
    coverage = coverage_match.group(1) if coverage_match else "PARTIAL"
    coverage_reason = (
        coverage_reason_match.group(1).strip()
        if coverage_reason_match
        else "Could not assess coverage."
    )

    packet = {
        "query": query,
        "evidence": evidence_list,
        "coverage": coverage,
        "coverage_reason": coverage_reason,
        "retrieval_score": score,
        "attempt_number": attempts
    }

    print(f"  → Coverage: {coverage} — {coverage_reason}")
    print(f"  → Evidence pieces found: {len(evidence_list)}")

    existing_packets = state.get("research_packets", [])
    return {
        "research_packets": existing_packets + [packet],
        "researcher_attempts": attempts,
        "follow_up_query": None,   # clear it — Writer will set a new one if needed
        "needs_more_research": False
    }

In [37]:
# the writer agent node

WRITER_SYSTEM_PROMPT = """You are a specialist financial report writer.
You will receive a topic and structured research evidence from an Apple FY2024 10-K analysis.

Your job is to write ONE polished report section covering this topic.

Rules:
- Use a clear heading (## Topic Name)
- Write 2-3 paragraphs of analysis grounded in the evidence
- Include a small data table if the evidence contains multiple figures
- Be specific — cite exact numbers and fiscal year references
- End with a one-sentence "Key Takeaway:"

After the report section, add a separate block:
NEEDS_MORE_RESEARCH: <YES or NO>
FOLLOW_UP_QUERY: <specific query to send back to the Researcher, or NONE>
REASON: <one sentence>
"""

def writer_node(state: MultiAgentState) -> dict:
    attempts = state.get("writer_attempts", 0) + 1
    print(f"\n[NODE: writer] Attempt {attempts}")

    # Gather all evidence from all research packets
    all_evidence = []
    for packet in state.get("research_packets", []):
        for ev in packet.get("evidence", []):
            all_evidence.append(
                f"[{ev['confidence']} confidence] {ev['fact']}"
            )

    evidence_text = "\n".join([f"• {e}" for e in all_evidence])

    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Topic: {state['topic']}\n\n"
                f"Research Evidence:\n{evidence_text}"
            )
        }],
        system=WRITER_SYSTEM_PROMPT
    )

    # Split report section from the metadata block
    parts = re.split(r"\nNEEDS_MORE_RESEARCH:", raw, maxsplit=1)
    report_section = parts[0].strip()

    needs_more = False
    follow_up = None

    if len(parts) > 1:
        meta_block = parts[1]
        needs_match = re.search(r"^\s*(YES|NO)", meta_block)
        follow_up_match = re.search(r"FOLLOW_UP_QUERY:\s*(.+)", meta_block)

        needs_more_raw = needs_match.group(1) if needs_match else "NO"
        needs_more = needs_more_raw == "YES"

        follow_up_raw = (
            follow_up_match.group(1).strip()
            if follow_up_match
            else "NONE"
        )
        follow_up = None if follow_up_raw == "NONE" else follow_up_raw

    print(f"  → Report section written ({len(report_section)} chars)")
    print(f"  → Needs more research: {needs_more}")
    if follow_up:
        print(f"  → Follow-up query: {follow_up[:80]}")

    existing_sections = state.get("report_sections", [])
    return {
        "report_sections": existing_sections + [report_section],
        "needs_more_research": needs_more,
        "follow_up_query": follow_up,
        "writer_attempts": attempts
    }

In [38]:
# the compile node

COMPILE_SYSTEM_PROMPT = """You are a senior financial editor.
You will receive multiple report sections covering different aspects of an Apple FY2024 10-K analysis.

Your job is to:
1. Write a 3-sentence Executive Summary at the top
2. Assemble all sections in a logical order beneath it
3. Add a "Sources" note at the bottom: "All data sourced from Apple Inc. Form 10-K, Fiscal Year 2024 (ended September 28, 2024)."

Output the complete, publication-ready report.
"""

def compile_node(state: MultiAgentState) -> dict:
    print(f"\n[NODE: compile]")
    sections = state.get("report_sections", [])
    print(f"  → Compiling {len(sections)} section(s) into final report")

    sections_text = "\n\n".join(sections)

    final_report = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Topic: {state['topic']}\n\n"
                f"Report Sections to Compile:\n\n{sections_text}"
            )
        }],
        system=COMPILE_SYSTEM_PROMPT
    )

    print(f"  → Final report compiled ({len(final_report)} chars)")
    return {"final_report": final_report}

In [39]:
# conditional edge after writer

def route_after_writer(
    state: MultiAgentState
) -> Literal["researcher_node", "compile_node"]:
    needs_more = state.get("needs_more_research", False)
    researcher_attempts = state.get("researcher_attempts", 0)
    max_attempts = state.get("max_researcher_attempts", 3)

    if needs_more and researcher_attempts < max_attempts and state.get("follow_up_query"):
        print(f"  → Edge: needs more research → researcher_node "
              f"(attempt {researcher_attempts + 1}/{max_attempts})")
        return "researcher_node"

    if needs_more and researcher_attempts >= max_attempts:
        print(f"  → Edge: max researcher attempts reached → compile_node anyway")
    else:
        print(f"  → Edge: Writer satisfied → compile_node")

    return "compile_node"

In [40]:
# Building the multi-agent graph

DB_PATH = "checkpoints/rag_checkpoints.db"


def build_multi_agent_graph(db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path, check_same_thread=False)
    checkpointer = SqliteSaver(conn)

    graph = StateGraph(MultiAgentState)

    graph.add_node("researcher_node", researcher_node)
    graph.add_node("writer_node", writer_node)
    graph.add_node("compile_node", compile_node)

    graph.set_entry_point("researcher_node")

    # Fixed edges
    graph.add_edge("researcher_node", "writer_node")
    graph.add_edge("compile_node", END)

    # Conditional edge: Writer → Researcher (feedback) OR Writer → Compile
    graph.add_conditional_edges(
        "writer_node",
        route_after_writer,
        {
            "researcher_node": "researcher_node",
            "compile_node": "compile_node"
        }
    )

    return graph.compile(checkpointer=checkpointer)


multi_agent_graph = build_multi_agent_graph()
print("✅ Multi-agent graph compiled")

try:
    print(multi_agent_graph.get_graph().draw_ascii())
except Exception:
    print("ASCII visualisation unavailable")

✅ Multi-agent graph compiled
ASCII visualisation unavailable


In [41]:
# The multi-Agent runner
def run_multi_agent_report(
    topic: str,
    max_researcher_attempts: int = 3
) -> MultiAgentState:
    global usage_tracker
    usage_tracker = TokenUsage()

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'='*60}")
    print(f"Topic: {topic}")
    print(f"Thread ID: {thread_id}")
    print(f"{'='*60}")

    initial_state: MultiAgentState = {
        "topic": topic,
        "report_sections_requested": [topic],
        "current_research_query": topic,
        "research_packets": [],
        "researcher_attempts": 0,
        "max_researcher_attempts": max_researcher_attempts,
        "report_sections": [],
        "needs_more_research": False,
        "follow_up_query": None,
        "writer_attempts": 0,
        "final_report": None,
        "finished_at": None,
    }

    final_state = multi_agent_graph.invoke(initial_state, config=config)
    final_state["finished_at"] = datetime.now().isoformat()

    usage_tracker.report()

    print(f"\n{'='*60}")
    print(f"✅ FINAL REPORT")
    print(f"{'='*60}")
    print(final_state["final_report"])
    print(f"\nResearcher ran {final_state['researcher_attempts']} time(s)")
    print(f"Writer ran {final_state['writer_attempts']} time(s)")
    print(f"{'='*60}\n")

    filename = (
        f"traces/day18_multiagent_"
        f"{thread_id[:8]}_"
        f"{datetime.now().strftime('%H%M%S')}.json"
    )
    with open(filename, "w") as f:
        json.dump(dict(final_state), f, indent=2)
    print(f"📁 Saved to {filename}")

    return final_state

In [42]:
# evaluation
# ── Report 1: Should complete in one Researcher/Writer cycle ─────────────
# iPhone revenue is well-covered in the 10-K — Writer should be satisfied
# immediately without requesting follow-up research
print("\n" + "#"*60)
print("REPORT 1: iPhone Revenue Analysis")
print("Expected: single Researcher/Writer cycle")
print("#"*60)

state1 = run_multi_agent_report(
    topic="Apple iPhone revenue performance in fiscal year 2024 "
          "including year-over-year comparison and share of total net sales"
)


# ── Report 2: Should trigger the feedback loop ───────────────────────────
# Services + geographic breakdown is a two-part topic — Writer will likely
# request follow-up research on geographic detail after the first cycle
print("\n" + "#"*60)
print("REPORT 2: Services Revenue and Geographic Breakdown")
print("Expected: Writer requests follow-up research on geographic detail")
print("#"*60)

state2 = run_multi_agent_report(
    topic="Apple Services segment revenue in FY2024 and how "
          "revenue is distributed across geographic operating segments",
    max_researcher_attempts=3
)


# ── Comparison: naive single-call vs multi-agent report ──────────────────
print("\n" + "#"*60)
print("COMPARISON: Naive single-call on the same topic as Report 2")
print("#"*60)

naive_answer = tracked_llm_call(
    messages=[{
        "role": "user",
        "content": (
            "What was Apple's Services revenue in FY2024 and how "
            "is revenue distributed across geographic segments? "
            "Answer from the Apple FY2024 10-K."
        )
    }],
    system=GENERATOR_SYSTEM_PROMPT
)
print(f"\nNaive answer:\n{naive_answer}")
print(f"\nMulti-agent report length: {len(state2['final_report'])} chars")
print(f"Naive answer length      : {len(naive_answer)} chars")


############################################################
REPORT 1: iPhone Revenue Analysis
Expected: single Researcher/Writer cycle
############################################################

Topic: Apple iPhone revenue performance in fiscal year 2024 including year-over-year comparison and share of total net sales
Thread ID: b6f6c7e3-ac15-42d2-affe-aeab29deab71

[NODE: researcher] Attempt 1
  → Query: Apple iPhone revenue performance in fiscal year 2024 including year-over-year co
  → Retrieved 4 chunks (top score: 0.859)
  → Coverage: COMPLETE — The evidence provides the FY2024 iPhone revenue amount, the YoY change versus 2023, and the proportion of total net sales, fully addressing the query.
  → Evidence pieces found: 3

[NODE: writer] Attempt 1
  → Report section written (1545 chars)
  → Needs more research: False
  → Edge: Writer satisfied → compile_node

[NODE: compile]
  → Compiling 1 section(s) into final report
  → Final report compiled (2198 chars)

📣 Token usage repo